# Analyze your own profile
**Compare yourself with yourself.** This notebook runs the project's within-creator analysis on one Instagram profile:
which words show up in your reels that beat your own recent baseline, and which show up in the ones that fall below it.

1. Collect your reels with `collection/collect_reels_snippet.js` (see `collection/COLLECTION_GUIDE_PT.md`) and put the JSON file in a folder.
2. Set `RAW_FOLDER` and `USERNAME` below and run all cells. Transcription is optional: without it the analysis uses captions only.

In [ ]:
import sys, re
from pathlib import Path
import numpy as np, pandas as pd
ROOT = Path.cwd().parent if Path.cwd().name == "notebook" else Path.cwd()
sys.path.insert(0, str(ROOT))
import config
from src.ingest import load_raw
from src.outcomes import compute_outcomes
from src.features import build_features, FEATURE_COLUMNS
from src.experiments import word_ranking
from scipy.stats import spearmanr

RAW_FOLDER = ROOT / "data" / "raw"          # folder with your <username>_reels.json
USERNAME = "leandrobranquinho"              # your handle, lowercase, without @

## 1. Load your reels and measure lift
*Lift* = log(views) minus the median log(views) of your 10 previous and 10 next reels. Positive lift means the reel beat
your own baseline at that moment, so follower growth and seasonality cancel out.

In [ ]:
reels = load_raw(RAW_FOLDER)
reels = reels[reels.username == USERNAME].assign(sector="mine")
tf = config.PROCESSED_DIR / "transcripts.parquet"
if tf.exists():
    reels = reels.merge(pd.read_parquet(tf).drop_duplicates("reel_id", keep="last"), on="reel_id", how="left")
df = compute_outcomes(reels, config.MIN_AGE_DAYS, config.BASELINE_WINDOW, config.LABEL_QUANTILE,
                      config.TRAIN_FRACTION, min_reels=30)
df = build_features(df, config.LEXICON_DIR / "anew_br.csv")
print(f"{len(df)} reels, {int(df.y_viral.eq(1).sum())} above and {int(df.y_viral.eq(0).sum())} below your baseline")
df.sort_values("lift", ascending=False)[["posted_at", "views", "lift", "opening"]].head(5)

## 2. Your viral words and your flop words
A logistic regression on TF-IDF features gives P(viral | word) for every word you use. Words are kept only when they
rank in your top 15 in at least 70% of bootstrap refits, so one lucky reel cannot put a word on the list.

In [ ]:
lab = df[df.y_viral.notna()]
words = word_ranking(lab["text_all"], lab["y_viral"], n_boot=100)
words.sort_values(["direction", "stability"], ascending=[False, False])

## 3. Copy devices against your lift
Spearman correlation between each device (question, numbers, "você", calls to action, emotion from ANEW-Br) and your lift.
With ~100 reels, |rho| below 0.2 is mostly noise: treat these as hints to test, not rules.

In [ ]:
rows = []
for f in FEATURE_COLUMNS:
    sub = df[[f, "lift"]].dropna()
    if sub[f].nunique() > 1:
        rho, p = spearmanr(sub[f], sub["lift"])
        rows.append({"feature": f, "rho": round(rho, 3), "p": round(p, 3), "share_of_reels": round(sub[f].mean(), 2)})
pd.DataFrame(rows).sort_values("rho")

## 4. Read it next to the sector cards
Open `cards/copy_cards.md`. Our main finding is that viral words rarely transfer between creators, even in the same
sector, so your own list above matters more than any generic rule. Write two reels using your top viral words and
compare their lift with your baseline after 14 days.